# 相手の未公開情報を推定する（型推定）

`MinimaxPlayer.estimate_opponent_team()` をオーバーライドして、相手ポケモンの見えていない技を推定値で補完する。

`TreeSearchPlayer` の相手推定フックは項目別に分かれており、`estimate_opponent_team()`（技・特性・アイテムの推定）と `estimate_opponent_selection()`（選出インデックスの推定）のどちらか必要な方だけをオーバーライドすればよい。入口フックの `estimate_opponent()` は既定でこの2つに委譲するテンプレートメソッドのため、この例のように項目別フックだけを書けば足りる。

これらのフックは、相手の合法手が未公開で空の初手だけでなく、探索の最上位（`choose_command()`）のたびに毎回呼ばれる（実対戦での型推定に相当）。観測（`battle`）はターンごとに再構築されるため、公開済みの技（`revealed`）を上書きせず、未公開スロットだけを補うように書く必要がある。

[▶ Google Colabで開く](https://colab.research.google.com/github/tmwork1/jpoke/blob/main/examples/02_tree_search/03_estimate_opponent.ipynb)

In [1]:
%pip install -q jpoke

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: C:\Users\tmtmh\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
from jpoke import Battle
from jpoke.players import MinimaxPlayer, RandomPlayer

In [3]:
class TypeEstimatingPlayer(MinimaxPlayer):
    """相手の未公開の技を固定の推測技で埋めるAI（estimate_opponent_team()の拡張例）。"""

    # 実戦では対戦データベースや使用率統計等から推定するが、ここでは固定の
    # 推測技のリストで最小例を示す
    GUESSED_MOVES = ["れいとうビーム", "10まんボルト", "だいちのちから", "きあいだま"]

    def estimate_opponent_team(self, battle: Battle) -> None:
        opponent = battle.opponent(self)
        active = battle.get_active(opponent)
        # battle.get_active() は情報隠蔽済みの観測のため、active.moves には
        # 公開済み（revealed）の技しか入っていない。公開済み分は上書きせず、
        # 標準的な技スロット数（4）に満たない分だけ固定の推測技で補う
        revealed_names = [move.name for move in active.moves]
        n_missing = 4 - len(revealed_names)
        if n_missing > 0:
            guesses = [name for name in self.GUESSED_MOVES if name not in revealed_names]
            active.set_moves(revealed_names + guesses[:n_missing])

In [4]:
ai_player = TypeEstimatingPlayer("GuessingAI", max_plies=1, max_nodes=50)
ai_player.add_pokemon("カビゴン", moves=["のしかかり", "じしん"])

opponent_player = RandomPlayer("Opponent")
opponent_player.add_pokemon("カメックス", moves=["ハイドロポンプ", "れいとうビーム"])

battle = Battle(ai_player, opponent_player, seed=1)
battle.start()

# 1ターン目: 相手の技は2本とも未公開。estimate_opponent_team が4本に満たない
# 分を固定の推測技で埋めるため、fallback() には委譲されず、推定を踏まえた
# 探索が行われる（nodes_expanded > 0になる）
battle.step()
revealed = [m.name for m in battle.get_active(opponent_player).moves if m.revealed]
print(f"1ターン目: nodes_expanded={ai_player.nodes_expanded}, 公開済みの技={revealed}")

# 2ターン目: ターン1で実際に使われた技（ハイドロポンプ）が公開される。
# estimate_opponent_team は公開済みの技を上書きせず、残りの未公開スロットだけを
# 推測技で補う（観測はターンごとに再構築されるため、この関数は毎ターン呼ばれ、
# そのたびに未公開分を補い直す）
battle.step()
revealed = [m.name for m in battle.get_active(opponent_player).moves if m.revealed]
print(f"2ターン目: nodes_expanded={ai_player.nodes_expanded}, 公開済みの技={revealed}")

battle.print_logs("all")

1ターン目: nodes_expanded=32, 公開済みの技=['ハイドロポンプ']


2ターン目: nodes_expanded=16, 公開済みの技=['ハイドロポンプ', 'れいとうビーム']
Turn 0 : GuessingAI :  : バトル開始
Turn 0 : GuessingAI : カビゴン : カビゴン 入場
Turn 0 : Opponent : カメックス : カメックス 入場
Turn 1 : GuessingAI : カビゴン : テラスタル化した（タイプ: ノーマル）
Turn 1 : Opponent : カメックス : ハイドロポンプ PP -1
Turn 1 : GuessingAI : カビゴン : HP -58 (177/235)
Turn 1 : GuessingAI : カビゴン : のしかかり PP -1
Turn 1 : Opponent : カメックス : HP -76 (78/154)
Turn 2 : Opponent : カメックス : れいとうビーム PP -1
Turn 2 : GuessingAI : カビゴン : HP -28 (149/235)
Turn 2 : GuessingAI : カビゴン : のしかかり PP -1
Turn 2 : Opponent : カメックス : HP -70 (8/154)
